# Linear Regression from Scratch: 2D Animation Workload Forecasting

**Series:** Machine Learning from Scratch | Part 3 of 3 -- Linear Regression, Evaluation, and Prediction

**Document Purpose.** This student notebook guides you through implementing Linear Regression from scratch with NumPy using the Normal Equation, validating results with scikit-learn, evaluating model performance, visualizing predictions, and deploying the model for inference. Complete all `???` placeholders and answer the logic questions based on the executed results.

**Context.** In Part 1, we cleaned and enriched the 2D character dataset `characters_clean.csv`. In Part 2, we performed exploratory data analysis and selected the five most predictive numerical features to produce `characters_features.csv`. In this final laboratory, our goal is to construct a predictive regression model that estimates the total `production_hours` required to animate and render a character based on its structural attributes (`num_frames_total`, `screen_time_ratio`, `num_states`, `fps`, `has_special_fx`).

**Learning Objectives:**
- Understand the mathematical formulation of Ordinary Least Squares (OLS) regression.
- Implement the Normal Equation $\beta = (X^T X)^{-1} X^T y$ from scratch using NumPy matrix operations.
- Compute core regression evaluation metrics ($R^2$, MAE, RMSE) from mathematical definitions.
- Train, validate, and interpret multiple linear regression models using `scikit-learn`.
- Split datasets into training and testing partitions (`train_test_split`) and evaluate generalization.
- Visualize actual vs. predicted values and diagnose residual behavior.
- Perform inference to forecast production hours for an unreleased character (`Nova`).

**Instructions.** Execute each cell sequentially. Cells marked `# Complete this code` contain `???` placeholders that you must replace with valid Python code.

---
## Setup: Importing Computational and Modeling Libraries

We import `pandas` for dataset loading, `numpy` for linear algebra matrix operations, `matplotlib.pyplot` and `seaborn` for diagnostics, and `scikit-learn` modules for model training and metric verification.

In [ ]:
# Setup -- execute this cell first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
print("Libraries loaded successfully.")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

---
## Exercise 3.0 -- Mathematical Foundations of Linear Regression

*(This is an informational theory cell. Read carefully before proceeding.)*

### 1. The Linear Model Formulation
Linear regression models the relationship between an observed continuous scalar response $y$ and an $n$-dimensional vector of explanatory features $\mathbf{x} = [x_1, x_2, \dots, x_n]^T$:
$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n$$
Where:
- $\beta_0$ is the **intercept** (bias term): the predicted value of $y$ when all predictor variables equal zero.
- $\beta_1, \dots, \beta_n$ are the **slopes** (coefficients): the expected change in $y$ associated with a one-unit increase in $x_j$, holding all other variables constant.

### 2. Matrix Representation
By augmenting the feature matrix with a leading column of ones (the bias column), we express the equation compactly for $m$ observations:
$$\mathbf{X}_b = \begin{bmatrix} 1 & x_{1,1} & \dots & x_{1,n} \\ 1 & x_{2,1} & \dots & x_{2,n} \\ \vdots & \vdots & \ddots & \vdots \\ 1 & x_{m,1} & \dots & x_{m,n} \end{bmatrix}, \quad \boldsymbol{\beta} = \begin{bmatrix} \beta_0 \\ \beta_1 \\ \vdots \\ \beta_n \end{bmatrix}, \quad \hat{\mathbf{y}} = \mathbf{X}_b \boldsymbol{\beta}$$

### 3. Cost Function: Mean Squared Error (MSE)
The model finds the coefficient vector $\boldsymbol{\beta}$ that minimizes the sum of squared differences between observed values $y_i$ and predicted values $\hat{y}_i$:
$$J(\boldsymbol{\beta}) = \text{MSE} = \frac{1}{m} \sum_{i=1}^m (y_i - \hat{y}_i)^2 = \frac{1}{m} \|\mathbf{X}_b \boldsymbol{\beta} - \mathbf{y}\|^2$$

### 4. The Normal Equation (Closed-Form Analytical Solution)
Setting the gradient $\nabla_{\boldsymbol{\beta}} J(\boldsymbol{\beta}) = \mathbf{0}$ yields the exact closed-form solution:
$$\boldsymbol{\beta} = (\mathbf{X}_b^T \mathbf{X}_b)^{-1} \mathbf{X}_b^T \mathbf{y}$$
Where:
- $\mathbf{X}_b^T$ is the transpose of $\mathbf{X}_b$.
- $(\mathbf{X}_b^T \mathbf{X}_b)^{-1}$ is the matrix inverse computed via `np.linalg.inv`.
- `@` is the matrix multiplication operator in Python.

**Why implement from scratch first?** Understanding that `scikit-learn` computes this exact algebraic operation eliminates the 'black box' mentality and empowers you to diagnose numerical instabilities (e.g. collinearity) in real-world pipelines.

---
## Exercise 3.1 -- Data Ingestion and Matrix Assembly

**Reference.** Tabular data must be converted into numerical NumPy arrays before matrix multiplication. The feature matrix $\mathbf{X}$ is extracted using `df[columns].values`, and the bias column is concatenated using `np.hstack([np.ones((m, 1)), X])`.

**Example.** The cell below illustrates assembling $\mathbf{X}_b$ on toy data.

In [ ]:
# Example -- do not modify
toy_data = pd.DataFrame({"x1": [1, 2, 3], "x2": [10, 20, 30], "y": [5, 9, 13]})
X_toy = toy_data[["x1", "x2"]].values
ones_col = np.ones((len(X_toy), 1))
Xb_toy = np.hstack([ones_col, X_toy])
print("X_toy shape :", X_toy.shape)
print("Xb_toy shape:", Xb_toy.shape)
print("Xb_toy matrix:\n", Xb_toy)

**Task.** Complete the student cell below:
1. Load `characters_features.csv` into a DataFrame named `df` using `pd.read_csv`.
2. Define the predictor feature list `feature_cols = ["num_frames_total", "screen_time_ratio", "num_states", "fps", "has_special_fx"]` and target `target_col = "production_hours"`.
3. Extract feature matrix `X` and target vector `y` as NumPy arrays (`.values`).
4. Create a column vector of ones of length `len(y)` with shape `(len(y), 1)`.
5. Stack the ones column and `X` horizontally to create `X_b` using `np.hstack`.
6. Print the shapes of `X`, `X_b`, and `y`.

In [ ]:
# Complete this code
df = pd.read_csv(???)
feature_cols = [???, ???, ???, ???, ???]
target_col = ???

X = df[feature_cols].???
y = df[target_col].???

ones = np.ones((len(y), ???))
X_b = np.hstack([???, X])

print("Dataset preview:")
print(df.head(3))
print(f"\nFeature matrix X shape: {X.shape}")
print(f"Bias matrix X_b shape : {X_b.shape}")
print(f"Target vector y shape : {y.shape}")

**Logic Question 3.1.** Based on the output of the cell above:

a) What are the exact dimensions of matrix `X_b`? Why does it have 6 columns instead of 5?
b) Why is appending a column of ones necessary for the matrix formulation of the Normal Equation?

---
## Exercise 3.2 -- Simple Linear Regression from Scratch (NumPy)

**Reference.** In simple linear regression with a single predictor $x$ (in our case, `num_frames_total`, the strongest individual predictor identified in Part 2), the model is:
$$\hat{y} = \beta_0 + \beta_1 x$$
The Normal Equation is solved using NumPy linear algebra:
`beta = np.linalg.inv(X.T @ X) @ X.T @ y`.

**Example.** The cell below computes simple linear regression coefficients for a synthetic series.

In [ ]:
# Example -- do not modify
x_demo = np.array([10, 20, 30, 40])
y_demo = np.array([25, 45, 65, 85])
X_demo = np.column_stack([np.ones(len(x_demo)), x_demo])
beta_demo = np.linalg.inv(X_demo.T @ X_demo) @ X_demo.T @ y_demo
print(f"Demo Intercept (beta_0): {beta_demo[0]:.2f}")
print(f"Demo Slope     (beta_1): {beta_demo[1]:.2f}")

**Task.** Complete the student cell below:
1. Extract `x_simple = df["num_frames_total"].values`.
2. Construct `X_simple = np.column_stack([np.ones(len(x_simple)), x_simple])`.
3. Compute `beta_simple` using the Normal Equation via `np.linalg.inv` and matrix multiplication `@`.
4. Print the intercept $\beta_0$ and slope $\beta_1$.
5. Compute predictions `y_pred_simple = X_simple @ beta_simple`.
6. Plot actual characters as scatter points and the regression line on the same figure.

In [ ]:
# Complete this code
x_simple = df["num_frames_total"].???
X_simple = np.column_stack([np.ones(???), x_simple])

beta_simple = np.linalg.???(X_simple.T @ X_simple) @ X_simple.T @ y

print(f"Intercept (beta_0): {beta_simple[0]:.4f}")
print(f"Slope     (beta_1): {beta_simple[1]:.4f}")
print(f"Interpretation: Each additional animation frame requires ~{beta_simple[1]:.2f} hours of production.")

y_pred_simple = ??? @ beta_simple

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(df["num_frames_total"], y, color="steelblue", label="Actual characters", s=60)
plt.plot(df["num_frames_total"], y_pred_simple, color="crimson", linewidth=2, label=f"Fit: y = {beta_simple[0]:.2f} + {beta_simple[1]:.2f}x")
plt.xlabel(???)
plt.ylabel(???)
plt.title("Simple Linear Regression: Frames vs. Production Hours")
plt.legend()
plt.tight_layout()
plt.show()

**Logic Question 3.2.** Based on the output of the cell above:

a) What are the exact numerical values of the intercept $\beta_0$ and slope $\beta_1$ calculated by the Normal Equation?
b) According to this simple model, how many production hours are predicted for a character requiring 120 animation frames?
c) What is the physical and artistic interpretation of the intercept $\beta_0 \approx 17.59$ hours?

---
## Exercise 3.3 -- Evaluation Metrics from Scratch (NumPy)

**Reference.** Three standard metrics quantify regression accuracy:
1. **Mean Absolute Error (MAE):** Average magnitude of errors without considering direction:
$$\text{MAE} = \frac{1}{m} \sum_{i=1}^m |y_i - \hat{y}_i|$$
2. **Root Mean Squared Error (RMSE):** Square root of average squared errors; penalizes large errors more heavily:
$$\text{RMSE} = \sqrt{\frac{1}{m} \sum_{i=1}^m (y_i - \hat{y}_i)^2}$$
3. **Coefficient of Determination ($R^2$):** Proportion of variance in $y$ explained by the model:
$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

**Example.** The cell below computes MAE, RMSE, and $R^2$ on sample arrays.

In [ ]:
# Example -- do not modify
y_act = np.array([100.0, 50.0, 80.0])
y_hat = np.array([95.0, 55.0, 75.0])
e = y_act - y_hat
print("MAE demo :", np.mean(np.abs(e)))
print("RMSE demo:", np.sqrt(np.mean(e**2)))
print("R2 demo  :", 1 - (np.sum(e**2) / np.sum((y_act - np.mean(y_act))**2)))

**Task.** Complete the student cell below:
1. Calculate the residuals: `residuos_simple = y - y_pred_simple`.
2. Compute `mae_simple` using `np.mean(np.abs(residuos_simple))`.
3. Compute `rmse_simple` using `np.sqrt(np.mean(residuos_simple**2))`.
4. Compute $\text{SS}_{\text{res}}$ and $\text{SS}_{\text{tot}}$ to obtain `r2_simple`.
5. Print the formatted metrics.

In [ ]:
# Complete this code
residuos_simple = y - y_pred_simple
mae_simple = np.mean(np.???(residuos_simple))
rmse_simple = np.???(np.mean(residuos_simple**???))
ss_res_simple = np.sum(residuos_simple**2)
ss_tot = np.sum((y - np.mean(y))**2)
r2_simple = 1 - (??? / ???)

print("Simple Linear Regression Evaluation (From Scratch):")
print(f"  MAE  (Mean Absolute Error)     : {mae_simple:.2f} hours")
print(f"  RMSE (Root Mean Squared Error) : {rmse_simple:.2f} hours")
print(f"  R^2  (Coefficient of Determ.)  : {r2_simple:.4f}")

**Logic Question 3.3.** Based on the output of the cell above:

a) What is the calculated $R^2$ value for the simple regression model? What proportion of variance does it explain?
b) Why is the RMSE ($10.30$ hours) higher than the MAE ($8.47$ hours)?
c) If a character has an actual value of 142.0 hours and a predicted value of 138.7 hours, what is its residual? Did the model overestimate or underestimate?

---
## Exercise 3.4 -- Multiple Linear Regression with scikit-learn

**Reference.** While simple regression uses one feature, multiple regression incorporates all 5 selected features:
$$\hat{y} = \beta_0 + \beta_1 x_{\text{frames}} + \beta_2 x_{\text{ratio}} + \beta_3 x_{\text{states}} + \beta_4 x_{\text{fps}} + \beta_5 x_{\text{fx}}$$
To evaluate real-world generalization, we partition the 20 characters into an 80% training set (16 characters) and a 20% test set (4 characters) using `train_test_split(..., test_size=0.20, random_state=42)`.

**Example.** The cell below illustrates training and splitting with scikit-learn.

In [ ]:
# Example -- do not modify
X_ex = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]])
y_ex = np.array([5, 11, 17, 23, 29])
X_tr, X_te, y_tr, y_te = train_test_split(X_ex, y_ex, test_size=0.2, random_state=42)
demo_model = LinearRegression().fit(X_tr, y_tr)
print("Model trained. Coefficients:", demo_model.coef_)

**Task.** Complete the student cell below:
1. Split `X` and `y` using `train_test_split` with `test_size=0.20` and `random_state=42`. Also track character indices.
2. Instantiate `model = LinearRegression()`.
3. Fit the model on the training set: `model.fit(X_train, y_train)`.
4. Predict on `X_test` and `X_train`.
5. Inspect the fitted intercept (`model.intercept_`) and coefficients (`model.coef_`).

In [ ]:
# Complete this code
indices = np.arange(len(df))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, indices, test_size=???, random_state=???
)

print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")
print(f"Test character names: {df.iloc[idx_test]["name"].tolist()}")

model = LinearRegression()
model.???(X_train, y_train)

y_pred_multi_test = model.???(X_test)
y_pred_multi_train = model.???(X_train)

print(f"\nModel Intercept (beta_0): {model.intercept_:.4f}")
print("Feature Coefficients (betas):")
for feat, coef in zip(feature_cols, model.???):
    print(f"  {feat:20s}: {coef:.4f}")

**Logic Question 3.4.** Based on the output of the cell above:

a) How many samples are allocated to training and testing? Which specific characters were placed in the test set?
b) What is the value and practical meaning of the coefficient for `has_special_fx` ($+4.7547$)?
c) Why is `random_state=42` used in the train/test split?

---
## Exercise 3.5 -- Multiple Model Evaluation and Benchmark Comparison

**Reference.** Evaluating regression models requires comparing performance across subsets (training vs testing) and between architectures (single-variable vs multi-variable) to diagnose overfitting or underfitting.
- If $R^2_{\text{train}} \gg R^2_{\text{test}}$, the model has overfitted to the training samples.
- `r2_score(y_true, y_pred)`, `mean_absolute_error(y_true, y_pred)`, `mean_squared_error(y_true, y_pred)` from `sklearn.metrics`.

**Example.** The cell below illustrates computing metrics using scikit-learn functions.

In [ ]:
# Example -- do not modify
y_t = np.array([100, 50, 80])
y_p = np.array([96, 52, 78])
print("R2 score :", r2_score(y_t, y_p))
print("MAE score:", mean_absolute_error(y_t, y_p))
print("RMSE score:", np.sqrt(mean_squared_error(y_t, y_p)))

**Task.** Complete the student cell below:
1. Compute `r2_multi_test`, `mae_multi_test`, and `rmse_multi_test` on `y_test` and `y_pred_multi_test`.
2. Compute `r2_multi_train`, `mae_multi_train`, and `rmse_multi_train` on `y_train` and `y_pred_multi_train`.
3. Print the benchmark comparison table comparing Simple (Full), Multiple (Train), and Multiple (Test).

In [ ]:
# Complete this code
r2_multi_test = r2_score(???, ???)
mae_multi_test = mean_absolute_error(???, ???)
rmse_multi_test = np.sqrt(mean_squared_error(???, ???))

r2_multi_train = r2_score(???, ???)
mae_multi_train = mean_absolute_error(???, ???)
rmse_multi_train = np.sqrt(mean_squared_error(???, ???))

print("=" * 60)
print(f"{'Metric':<22} {'Simple (Full)':>12} {'Multiple (Train)':>18} {'Multiple (Test)':>16}")
print("=" * 60)
print(f"{'R^2':<22} {r2_simple:>12.4f} {r2_multi_train:>18.4f} {r2_multi_test:>16.4f}")
print(f"{'MAE (hours)':<22} {mae_simple:>12.2f} {mae_multi_train:>18.2f} {mae_multi_test:>16.2f}")
print(f"{'RMSE (hours)':<22} {rmse_simple:>12.2f} {rmse_multi_train:>18.2f} {rmse_multi_test:>16.2f}")
print("=" * 60)

**Logic Question 3.5.** Based on the benchmark comparison table above:

a) Compare $R^2$ on the training set ($0.9929$) versus the test set ($0.8580$). Why is performance higher on the training set?
b) Why should instructors and data scientists be cautious about concluding that the model has high test error when test size is $N=4$?

---
## Exercise 3.6 -- Visualization: Actual vs. Predicted with Character Annotations

**Reference.** An Actual vs. Predicted scatter plot displays ground truth on the horizontal axis ($x$) and model predictions on the vertical axis ($y$). The identity line $y = x$ represents perfect prediction. Points lying above the line were overestimated; points below were underestimated.

**Example.** The cell below illustrates generating an annotated prediction plot.

In [ ]:
# Example -- do not modify
y_true_ex = [50, 100, 150]
y_pred_ex = [48, 104, 145]
labels_ex = ["A", "B", "C"]
plt.figure(figsize=(5, 4))
plt.scatter(y_true_ex, y_pred_ex, color="teal")
plt.plot([40, 160], [40, 160], "r--")
for i, txt in enumerate(labels_ex):
    plt.annotate(txt, (y_true_ex[i], y_pred_ex[i]))
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.show()

**Task.** Complete the student cell below:
1. Extract test character names using `df.iloc[idx_test]["name"].values`.
2. Create a scatter plot of `y_test` vs `y_pred_multi_test`.
3. Annotate each scatter point with the character's name using `plt.annotate`.
4. Draw the red dashed diagonal line of perfect prediction ($y = x$).
5. Print the individual prediction breakdown table.

In [ ]:
# Complete this code
nombres_test = df.iloc[idx_test]["name"].???

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_multi_test, color="royalblue", s=80, edgecolors="black", label="Test characters")

for i, name in enumerate(nombres_test):
    plt.annotate(f" {name}", (y_test[i], y_pred_multi_test[i]), fontsize=10, fontweight="bold")

min_val = min(y_test.min(), y_pred_multi_test.min()) - 10
max_val = max(y_test.max(), y_pred_multi_test.max()) + 10
plt.plot([min_val, max_val], [min_val, max_val], "r--", label="Perfect prediction (y = x)")

plt.xlabel(???)
plt.ylabel(???)
plt.title("Multiple Regression: Actual vs. Predicted Hours (Test Set)")
plt.legend()
plt.tight_layout()
plt.show()

print("Individual test set predictions:")
for i, name in enumerate(nombres_test):
    error = y_pred_multi_test[i] - y_test[i]
    print(f"  {name:10s} -> Actual: {y_test[i]:6.1f} h | Predicted: {y_pred_multi_test[i]:6.1f} h | Error: {error:+6.1f} h")

**Logic Question 3.6.** Based on the plot and individual prediction table above:

a) What does a point lying directly on the dashed red line $y = x$ signify?
b) For which specific test character did the model achieve the lowest absolute prediction error?
c) Which test character experienced the largest underestimation error?

---
## Exercise 3.7 -- Residual Analysis

**Reference.** Residual analysis investigates model assumptions:
- **Homoscedasticity:** Residuals should be randomly distributed around zero with constant variance across the range of fitted values.
- **Curvature/Patterns:** A curved residual trend (e.g. U-shape) indicates an unmodeled non-linear relationship.
- `residuos = y_actual - y_pred` plotted against `y_pred` with a horizontal reference line at $y = 0$.

**Example.** The cell below illustrates generating a residual plot.

In [ ]:
# Example -- do not modify
p_ex = np.array([40, 80, 120, 160])
r_ex = np.array([2, -3, 4, -1])
plt.figure(figsize=(5, 3))
plt.scatter(p_ex, r_ex, color="coral")
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Fitted Values"); plt.ylabel("Residuals"); plt.show()

**Task.** Complete the student cell below:
1. Compute test set residuals: `residuos_test = y_test - y_pred_multi_test`.
2. Plot a scatter plot of `y_pred_multi_test` on the x-axis vs `residuos_test` on the y-axis.
3. Add a dashed horizontal reference line at $y = 0$ using `plt.axhline`.
4. Annotate each residual point with its character name.
5. Print the mean and standard deviation of the residuals.

In [ ]:
# Complete this code
residuos_test = ??? - ???

plt.figure(figsize=(8, 4))
plt.scatter(y_pred_multi_test, residuos_test, color="coral", s=80, edgecolors="black")
plt.axhline(y=0, color="black", linestyle="--", linewidth=1.5, label="Residual = 0")
for i, name in enumerate(nombres_test):
    plt.annotate(f" {name}", (y_pred_multi_test[i], residuos_test[i]), fontsize=10)

plt.xlabel("Predicted Values (y_pred)")
plt.ylabel("Residuals (y_actual - y_pred)")
plt.title("Residual Analysis — Test Set")
plt.legend()
plt.tight_layout()
plt.show()

print("Residual values summary:")
print(f"  Mean residual: {np.mean(residuos_test):.2f} hours")
print(f"  Std residual : {np.std(residuos_test):.2f} hours")

**Logic Question 3.7.** Based on the residual plot above:

a) What does a horizontal residual band evenly dispersed around 0 signify in linear regression?
b) What would a systematic funnel shape (widening residuals as $\hat{y}$ increases) signify?

---
## Exercise 3.8 -- Production Forecasting for a New Character (Inference)

**Reference.** Once trained and validated, a machine learning model is deployed to forecast workloads for upcoming characters before production begins. New inputs must be provided as a 2D array matching the exact feature order used during training: `model.predict([[feat1, feat2, ...]])`.

**Scenario:** The studio is planning season 2 of *Neon Skies* and has designed an unreleased cybernetic hero named **Nova**:
- `num_frames_total`: 96 frames
- `screen_time_ratio`: 0.20
- `num_states`: 4 states
- `fps`: 24 fps
- `has_special_fx`: 1 (energy blade visual effects)

In [ ]:
# Example -- do not modify
sample_char = np.array([[60, 0.15, 3, 24, 0]])
est_hours = model.predict(sample_char)
print(f"Sample character estimate: {est_hours[0]:.1f} hours")

**Task.** Complete the student cell below:
1. Define the dictionary for `nova` with the specifications listed above.
2. Construct the 2D feature array `X_nova = np.array([[nova[col] for col in feature_cols]])`.
3. Generate the workload forecast using `model.predict(X_nova)`.
4. Perform a sensitivity check by re-predicting Nova's hours with `has_special_fx = 0` to compute the workload added by special effects.

In [ ]:
# Complete this code
nova = {
    "num_frames_total": ???,
    "screen_time_ratio": ???,
    "num_states": ???,
    "fps": ???,
    "has_special_fx": ???
}

X_nova = np.array([[nova[col] for col in ???]])
pred_nova = model.???(X_nova)

print(f"Estimated production hours for Nova: {pred_nova[0]:.2f} hours")
print(f"(Reference: Studio dataset average is {df['production_hours'].mean():.2f} hours)")

# Sensitivity check: what if Nova does not have special FX?
nova_no_fx = X_nova.copy()
nova_no_fx[0, feature_cols.index("has_special_fx")] = 0
pred_nova_no_fx = model.predict(nova_no_fx)
fx_impact = pred_nova[0] - pred_nova_no_fx[0]
print(f"Estimated production hours without special FX: {pred_nova_no_fx[0]:.2f} hours")
print(f"Workload impact of special FX: {fx_impact:.2f} hours")

**Logic Question 3.8.** Based on the inference output above:

a) What is the forecasted production time for Nova? How many standard 40-hour work weeks does this represent?
b) How many hours of studio work does the inclusion of special visual effects (`has_special_fx = 1`) add to Nova's pipeline?
c) Why is Nova's estimate (120.97 h) higher than the studio average of 96.32 hours?

---
## Exercise 3.9 -- Executive Summary Report

**Task.** Execute the cell below to print the official studio production benchmark report synthesizing all modeling achievements.

In [ ]:
# Executive Summary -- do not modify
print("=" * 60)
print("     STUDIO PRODUCTION FORECAST REPORT — NEON SKIES")
print("=" * 60)
print(f"  Roster Size         : {len(df)} characters")
print(f"  Predictive Features : {len(feature_cols)} ({.join(feature_cols)})")
print(f"  Simple Model (1-Var): R^2 = {r2_simple:.4f} | RMSE = {rmse_simple:.2f} h")
print(f"  Multi Model (Train) : R^2 = {r2_multi_train:.4f} | RMSE = {rmse_multi_train:.2f} h")
print(f"  Multi Model (Test)  : R^2 = {r2_multi_test:.4f} | RMSE = {rmse_multi_test:.2f} h")
print(f"  Forecast for Nova   : {pred_nova[0]:.1f} hours ({pred_nova[0]/40:.1f} work weeks)")
print("=" * 60)

     STUDIO PRODUCTION FORECAST REPORT — NEON SKIES
  Roster Size         : 20 characters
  Predictive Features : 5 (num_frames_total, screen_time_ratio, num_states, fps, has_special_fx)
  Simple Model (1-Var): R^2 = 0.9569 | RMSE = 10.30 h
  Multi Model (Train) : R^2 = 0.9929 | RMSE = 3.69 h
  Multi Model (Test)  : R^2 = 0.8580 | RMSE = 23.42 h
  Forecast for Nova   : 121.0 hours (3.0 work weeks)
